In [ ]:
import sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')

ISI_CONFIG = '''from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
AKAR = ROOT / 'KTI_KARHUTLA'
HASIL = AKAR / 'hasil'
LAIN = AKAR / 'lainnya'

GADM_SHP = ROOT / 'GADM 2' / 'gadm41_IDN_2.shp'
GAMBUT_GEOJSON = LAIN / 'Indonesia_Peat_Lands.geojson'

TABEL_UTAMA = HASIL / 'tabel_gabungan_nasional_semua_bulan.csv'
PROB_FINAL = HASIL / 'prob_final_nasional.npz'
GAMBUT_KAB = HASIL / 'gambut_kabkota.csv'
VALIDASI_SIPONGI = HASIL / 'validasi_sipongi_firms_kabtahun.csv'
FIRMS_GADM = AKAR / 'firms_gadm'
FIRMS_BERSIH = HASIL / 'firms_nasional_harian_bersih.csv'
PRIORITAS_BERSIH = HASIL / 'prioritas_harian_bersih.csv'
HASIL_BACKTEST = HASIL / 'backtest_skor_harian.csv'
PRIORITAS_HARIAN = HASIL / 'prioritas_harian.csv'

PRIORITAS = ['Riau', 'Jambi', 'Sumatera Selatan',
             'Kalimantan Barat', 'Kalimantan Tengah', 'Kalimantan Selatan']
CRS_LUAS = 'EPSG:6933'
TAU = 0.05
'''
(ROOT / 'config_ui.py').write_text(ISI_CONFIG)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import config_ui as cfg

Mounted at /content/drive


In [ ]:
kolom = ['acq_date', 'confidence', 'type', 'GID_2', 'frp', 'latitude', 'longitude', 'version']
fr = pd.concat([pd.read_csv(p, usecols=kolom) for p in sorted(cfg.FIRMS_GADM.glob('viirs_*.csv'))],
               ignore_index=True)
fr['tanggal'] = pd.to_datetime(fr['acq_date'])
fr = fr[fr['tanggal'] >= '2020-01-01'].copy()
fr['nrt'] = fr['version'].astype(str).str.contains('NRT')
fr['rendah'] = fr['confidence'].astype(str).str.lower() == 'l'
fr['sel'] = (np.floor(fr['latitude'] * 100).astype('int64') * 100000
             + np.floor(fr['longitude'] * 100).astype('int64'))
print(fr.shape, '| NRT:', int(fr['nrt'].sum()), '| SP:', int((~fr['nrt']).sum()))

(1203383, 12) | NRT: 502198 | SP: 701185


In [ ]:
HARI_MIN = 50
TAHUN_MIN = 3

sp = fr[~fr['nrt']]
sel_tipe = set(sp.loc[sp['type'].isin([1, 2, 3]), 'sel'])

hari_sel = (sp[~sp['rendah']]
            .assign(th=lambda d: d['tanggal'].dt.year)
            .groupby(['sel', 'th'])['tanggal'].nunique()
            .reset_index(name='hari'))
sel_ulang = set(hari_sel[hari_sel['hari'] >= HARI_MIN].groupby('sel')['th'].nunique()
                .loc[lambda s: s >= TAHUN_MIN].index)

masker = sel_tipe | sel_ulang
fr['statis'] = fr['sel'].isin(masker)
print('sel bertipe statis:', len(sel_tipe), '| sel berulang:', len(sel_ulang), '| total masker:', len(masker))

tipe0 = sp[(sp['type'] == 0) & ~sp['rendah']]
print('titik SP tipe 0 yang ikut terbuang oleh masker:',
      round(100 * tipe0['sel'].isin(masker).mean(), 2), '%')

sel bertipe statis: 1671 | sel berulang: 60 | total masker: 1671
titik SP tipe 0 yang ikut terbuang oleh masker: 3.8 %


In [ ]:
d = fr[~fr['rendah'] & (fr['tanggal'] >= '2026-01-01')].copy()
d['bln'] = d['tanggal'].dt.to_period('M')
bulan = d.groupby('bln').agg(titik=('frp', 'size'),
                             hari=('tanggal', 'nunique'),
                             persen_NRT=('nrt', 'mean'),
                             titik_sel_statis=('statis', 'sum'))
bulan['statis_per_hari'] = bulan['titik_sel_statis'] / bulan['hari']
bulan['persen_statis'] = bulan['titik_sel_statis'] / bulan['titik']
print(bulan.round(3).to_string())

m = fr[~fr['rendah'] & ~fr['nrt'] & fr['tanggal'].dt.month.between(7, 10)]
tahun = m.groupby(m['tanggal'].dt.year).agg(hari=('tanggal', 'nunique'), statis=('statis', 'sum'))
print('\nPEMBANDING SP, Juli sampai Oktober, titik di sel statis per hari')
print((tahun['statis'] / tahun['hari']).round(1).to_string())

          titik  hari  persen_NRT  titik_sel_statis  statis_per_hari  persen_statis
bln                                                                                
2026-01    5618    31         0.0              1113           35.903          0.198
2026-02    3153    28         0.0               676           24.143          0.214
2026-03    6773    28         0.0               904           32.286          0.133
2026-04    4267    27         0.0              1587           58.778          0.372
2026-05    4162    31         0.0              1373           44.290          0.330
2026-06    7311    28         0.0              2012           71.857          0.275
2026-07   20982    25         1.0              1877           75.080          0.089
2026-08  138239    30         1.0              4072          135.733          0.029
2026-09  202590    30         1.0              3845          128.167          0.019
2026-10  114789    14         1.0              1504          107.429        

In [ ]:
def agregat(x):
    return (x.groupby(['tanggal', 'GID_2'])
             .agg(titik_panas=('frp', 'size'), frp_total=('frp', 'sum'))
             .reset_index())


dasar = fr[(fr['type'].fillna(0) == 0) & ~fr['rendah']]
bersih = agregat(dasar[~dasar['statis']])
bersih.to_csv(cfg.FIRMS_BERSIH, index=False)

asli = agregat(dasar)
tambah = asli.merge(bersih, on=['tanggal', 'GID_2'], how='left', suffixes=('', '_bersih')).fillna(0.0)
tambah['bln'] = tambah['tanggal'].dt.to_period('M')
susut = tambah[tambah['tanggal'] >= '2026-01-01'].groupby('bln')[['titik_panas', 'titik_panas_bersih']].sum()
susut['persen_dibuang'] = 1 - susut['titik_panas_bersih'] / susut['titik_panas']
print(susut.round(3).to_string())

         titik_panas  titik_panas_bersih  persen_dibuang
bln                                                     
2026-01         4616              4505.0           0.024
2026-02         2546              2477.0           0.027
2026-03         6025              5869.0           0.026
2026-04         2840              2680.0           0.056
2026-05         2934              2789.0           0.049
2026-06         5553              5299.0           0.046
2026-07        20982             19105.0           0.089
2026-08       138239            134167.0           0.029
2026-09       202590            198745.0           0.019
2026-10       114789            113285.0           0.013


In [ ]:
utama = pd.read_csv(cfg.TABEL_UTAMA, parse_dates=['tanggal'], usecols=['tanggal', 'GID_2', 'NAME_1', 'titik_panas'])
t = utama.merge(bersih[['tanggal', 'GID_2', 'titik_panas']].rename(columns={'titik_panas': 'tp_bersih'}),
                on=['tanggal', 'GID_2'], how='left')
t['tp_bersih'] = np.where(t['titik_panas'].isna(), np.nan, t['tp_bersih'].fillna(0.0))
t = t.sort_values(['GID_2', 'tanggal']).reset_index(drop=True)
t['api_7h_bersih'] = (t.groupby('GID_2')['tp_bersih']
                      .transform(lambda s: s.shift(1).rolling(7, min_periods=4).sum()))

pri = pd.read_csv(cfg.PRIORITAS_HARIAN, parse_dates=['tanggal'])
t = t.merge(pri[['tanggal', 'GID_2', 'p_K1', 'persen_gambut', 'skor_tanpa_gambut', 'skor']],
            on=['tanggal', 'GID_2'], how='left')

per_kab = t.drop_duplicates('GID_2').set_index('GID_2')['persen_gambut']
t['r_gambut'] = t['GID_2'].map(per_kab.rank(pct=True))
t['r_awan'] = t.groupby('tanggal')['p_K1'].rank(pct=True)
t['r_api_bersih'] = t.groupby('tanggal')['api_7h_bersih'].rank(pct=True)
t['skor_tanpa_gambut_bersih'] = t['r_awan'] * t['r_api_bersih']
t['skor_bersih'] = t['skor_tanpa_gambut_bersih'] * t['r_gambut']

simpan = ['tanggal', 'GID_2', 'NAME_1', 'p_K1', 'tp_bersih', 'api_7h_bersih', 'persen_gambut',
          'skor_tanpa_gambut_bersih', 'skor_bersih']
t[simpan].rename(columns={'tp_bersih': 'titik_panas_bersih'}).to_csv(
    cfg.PRIORITAS_BERSIH, index=False, float_format='%.4f')


def irisan_harian(x, a, b, k=10):
    hasil = {}
    for tgl, g in x.groupby('tanggal'):
        hasil[tgl] = len(set(g.nlargest(k, a)['GID_2']) & set(g.nlargest(k, b)['GID_2'])) / k
    return pd.Series(hasil)


sp26 = t[(t['tanggal'] >= '2026-01-08') & (t['tanggal'] < '2026-07-08')]
nrt26 = t[t['tanggal'] >= '2026-07-08']
for nama, a, b in [('tanpa gambut', 'skor_tanpa_gambut', 'skor_tanpa_gambut_bersih'),
                   ('dengan gambut', 'skor', 'skor_bersih')]:
    print(f'irisan top-10 lama vs bersih, {nama}: SP Jan-Jun 2026 =',
          round(irisan_harian(sp26, a, b).mean(), 3),
          '| NRT Jul 2026 sampai akhir =', round(irisan_harian(nrt26, a, b).mean(), 3))

hari = t.loc[t['api_7h_bersih'].notna(), 'tanggal'].max()
h = t[t['tanggal'] == hari]
tampil = ['GID_2', 'NAME_1', 'api_7h_bersih', 'persen_gambut']
print('\nhari terakhir:', hari.date())
print('TOP 10 LAMA, tanpa gambut:', h.nlargest(10, 'skor_tanpa_gambut')['GID_2'].tolist())
print('TOP 10 BERSIH, tanpa gambut:', h.nlargest(10, 'skor_tanpa_gambut_bersih')['GID_2'].tolist())
print('\nTOP 10 BERSIH, dengan gambut')
print(h.nlargest(10, 'skor_bersih')[tampil + ['skor_bersih']].round(3).to_string(index=False))

irisan top-10 lama vs bersih, tanpa gambut: SP Jan-Jun 2026 = 0.931 | NRT Jul 2026 sampai akhir = 0.945
irisan top-10 lama vs bersih, dengan gambut: SP Jan-Jun 2026 = 0.923 | NRT Jul 2026 sampai akhir = 0.955

hari terakhir: 2026-10-14
TOP 10 LAMA, tanpa gambut: ['IDN.22.10_1', 'IDN.18.5_1', 'IDN.22.1_1', 'IDN.18.7_1', 'IDN.22.7_1', 'IDN.18.3_1', 'IDN.18.4_1', 'IDN.23.8_1', 'IDN.23.28_1', 'IDN.19.10_1']
TOP 10 BERSIH, tanpa gambut: ['IDN.22.10_1', 'IDN.18.5_1', 'IDN.22.1_1', 'IDN.18.7_1', 'IDN.22.7_1', 'IDN.18.4_1', 'IDN.23.8_1', 'IDN.18.3_1', 'IDN.23.28_1', 'IDN.19.10_1']

TOP 10 BERSIH, dengan gambut
      GID_2            NAME_1  api_7h_bersih  persen_gambut  skor_bersih
IDN.22.10_1       Papua Barat          479.0         21.351        0.821
 IDN.22.7_1       Papua Barat          268.0         46.004        0.797
 IDN.22.1_1       Papua Barat         1023.0          6.036        0.711
 IDN.22.2_1       Papua Barat          842.0          6.591        0.638
 IDN.8.10_1             J

In [ ]:
PAKAI_BERSIH = False

kolom_p = ['tanggal', 'GID_2', 'p_K1', 'api_7h', 'skor_tanpa_gambut', 'skor']
if PAKAI_BERSIH:
    p = pd.read_csv(cfg.PRIORITAS_BERSIH, parse_dates=['tanggal'])
    p = p.rename(columns={'api_7h_bersih': 'api_7h', 'skor_tanpa_gambut_bersih': 'skor_tanpa_gambut',
                          'skor_bersih': 'skor', 'titik_panas_bersih': 'titik_panas'})
    p = p[['tanggal', 'GID_2', 'p_K1', 'api_7h', 'skor_tanpa_gambut', 'skor', 'titik_panas']]
    asal = pd.read_csv(cfg.TABEL_UTAMA, parse_dates=['tanggal'],
                       usecols=['tanggal', 'GID_2', 'hujan_rata', 'frac_hujan'])
    d = p.merge(asal, on=['tanggal', 'GID_2'], how='left')
else:
    p = pd.read_csv(cfg.PRIORITAS_HARIAN, parse_dates=['tanggal'], usecols=kolom_p)
    asal = pd.read_csv(cfg.TABEL_UTAMA, parse_dates=['tanggal'],
                       usecols=['tanggal', 'GID_2', 'titik_panas', 'hujan_rata', 'frac_hujan'])
    d = p.merge(asal, on=['tanggal', 'GID_2'], how='left')

d['k1'] = ((d['hujan_rata'] * 6) < 1) & (d['frac_hujan'] >= cfg.TAU)


def matriks(kol):
    return d.pivot(index='tanggal', columns='GID_2', values=kol)


P = matriks('titik_panas').sort_index()
fut = P.iloc[::-1].shift(1).rolling(7, min_periods=5).sum().iloc[::-1]
F = np.log1p(fut.to_numpy())
K = matriks('k1').reindex(index=P.index, columns=P.columns).to_numpy().astype(float)

layak = (np.isfinite(F).mean(axis=1) > 0.9) & (P.index.month >= 6) & (P.index.month <= 10)
layak &= matriks('api_7h').reindex(index=P.index).notna().mean(axis=1).to_numpy() > 0.9
print('hari dipakai:', int(layak.sum()), '| tahun:', sorted(set(P.index[layak].year)))

hari dipakai: 978 | tahun: [2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [ ]:
rng = np.random.default_rng(1)
acak = rng.random(P.shape)
metode = {
    'skor (dengan gambut)': matriks('skor'),
    'skor (tanpa gambut)': matriks('skor_tanpa_gambut'),
    'hanya api_7h': matriks('api_7h'),
    'hanya p_K1': matriks('p_K1'),
    'acak': pd.DataFrame(acak, index=P.index, columns=P.columns),
}


def top10(S):
    S = S.reindex(index=P.index, columns=P.columns).to_numpy()
    S = np.where(np.isnan(S), -1.0, S) + rng.random(S.shape) * 1e-9
    return np.argpartition(-S, 10, axis=1)[:, :10]


harian = {}
for nama, S in metode.items():
    idx = top10(S)
    ambil = lambda M: np.take_along_axis(M, idx, 1)
    harian[nama] = pd.DataFrame({'api_depan': ambil(F).mean(axis=1),
                                 'K1': ambil(K).mean(axis=1),
                                 'U': ambil(F * K).mean(axis=1)}, index=P.index)[layak]

ringkas = pd.DataFrame({n: h.mean() for n, h in harian.items()}).T
print(ringkas.round(3).sort_values('U', ascending=False).to_string())

pd.concat(harian, axis=1).to_csv(cfg.HASIL_BACKTEST, float_format='%.4f')

                      api_depan     K1      U
skor (tanpa gambut)       2.494  0.410  0.985
skor (dengan gambut)      2.370  0.387  0.866
hanya api_7h              3.897  0.194  0.692
hanya p_K1                0.779  0.483  0.368
acak                      0.914  0.219  0.177


In [ ]:
def bootstrap_blok(x, panjang=14, ulang=2000):
    x = np.asarray(x)
    n = len(x)
    banyak = int(np.ceil(n / panjang))
    rata = []
    for _ in range(ulang):
        mulai = rng.integers(0, n - panjang + 1, banyak)
        rata.append(np.concatenate([x[i:i + panjang] for i in mulai])[:n].mean())
    return np.percentile(rata, [2.5, 97.5])


baris = []
for a, b in [('skor (tanpa gambut)', 'hanya api_7h'), ('skor (dengan gambut)', 'hanya api_7h'),
             ('skor (tanpa gambut)', 'hanya p_K1'), ('skor (dengan gambut)', 'skor (tanpa gambut)')]:
    selisih = harian[a]['U'] - harian[b]['U']
    lo, hi = bootstrap_blok(selisih.to_numpy())
    baris.append({'selisih U': f'{a} - {b}', 'rata2': selisih.mean(), 'bk95_bawah': lo, 'bk95_atas': hi})
print(pd.DataFrame(baris).round(4).to_string(index=False))

per_tahun = pd.DataFrame({n: h['U'].groupby(h.index.year).mean() for n, h in harian.items()})
print('\nU per tahun')
print(per_tahun.round(3).to_string())

                                 selisih U   rata2  bk95_bawah  bk95_atas
        skor (tanpa gambut) - hanya api_7h  0.2922      0.2222     0.3595
       skor (dengan gambut) - hanya api_7h  0.1738      0.1057     0.2421
          skor (tanpa gambut) - hanya p_K1  0.6171      0.5435     0.6918
skor (dengan gambut) - skor (tanpa gambut) -0.1184     -0.1459    -0.0901

U per tahun
         skor (dengan gambut)  skor (tanpa gambut)  hanya api_7h  hanya p_K1   acak
tanggal                                                                            
2020                    0.756                0.880         0.555       0.325  0.164
2021                    0.761                0.861         0.563       0.317  0.139
2022                    0.562                0.695         0.602       0.267  0.138
2023                    1.112                1.219         0.787       0.427  0.176
2024                    0.904                1.053         0.739       0.358  0.173
2025                    0.833